# Ethnic Clothing Part Segmentation — Pipeline Overview

This notebook walks through every stage of the dataset build and shows what each stage produced:
how many images came in and went out, why images were removed, and examples of the results.
It only **reads** project files (`data/manifest.db`, `data/raw/*/log.jsonl`, `data/work/`, `dataset/`);
it never changes them. Sections whose stage has not run yet say so instead of failing.

| # | Stage | Code | Runs on |
|---|---|---|---|
| 1 | Search, source-page check, download | `modal_collect.py`, `ethnoseg/web.py` | Modal CPU |
| 2 | Ingest + screening (quality, near-duplicates) | `ethnoseg/ingest.py`, `ethnoseg/screen.py` | laptop |
| 3 | SAM 3 subject + part masks | `modal_segment.py`, `ethnoseg/sam_backend.py`, `ethnoseg/parts.py` | Modal GPU (L4) |
| 4 | White background, centring, 512×512, final masks | `ethnoseg/compose.py` | laptop |
| 5 | Human review (keep / reject) | `ethnoseg/review.py` | laptop, by a person |
| 6 | Split, export, QC | `ethnoseg/split.py`, `export.py`, `qc.py` | laptop |


In [ ]:
# Setup: run from the project folder
import json, os, sys, sqlite3, platform
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd()
if not (ROOT / "ethnoseg").exists() and (ROOT.parent / "ethnoseg").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from ethnoseg import settings, imgops

S = settings.load(os.environ.get("ETHNOSEG_CONFIG"))
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_colwidth", 80)

GROUPS = S.groups()
CODES = [g["code"] for g in GROUPS]
LABEL = {g["code"]: f"{g['name']} {g['zh']}" for g in GROUPS}
CLASSES = {c["id"]: c["name"] for c in S.classes}
MISSING = []          # image files an example cell wanted but could not find

def show_grid(paths_titles, cols=4, size=3.0, title=None, aspect=1.05):
    """Show images in a grid; missing files are listed in MISSING instead of failing."""
    items = [(p, t) for p, t in paths_titles]
    if not items:
        print("(no examples)")
        return
    rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size * aspect))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, (p, t) in zip(axes, items):
        if isinstance(p, np.ndarray):
            ax.imshow(p)
        elif Path(p).exists():
            ax.imshow(Image.open(p).convert("RGB"))
        else:
            MISSING.append(str(p))
            ax.text(0.5, 0.5, "file not on this computer", ha="center", va="center", fontsize=8)
        ax.set_title(t, fontsize=8)
    if title:
        fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    plt.show()

def manifest():
    if not S.db_path.exists():
        return None
    with sqlite3.connect(S.db_path) as con:
        return pd.read_sql_query("SELECT * FROM images", con)

M = manifest()
print("Project:", S.raw["project"]["name"])
print("Groups:", ", ".join(LABEL[c] for c in CODES))
print("Manifest:", "found" if M is not None else "not created yet", "-", 0 if M is None else len(M), "images")


## 0. Configuration
All stages read `config.yaml`. The tables below are the settings this build used.

In [ ]:
display(pd.DataFrame([{"code": g["code"], "group": g["name"], "zh": g["zh"],
                       "search terms": len(g["queries"]), "examples": ", ".join(g["queries"][:3])} for g in GROUPS]))
display(pd.DataFrame([{"id": c["id"], "class": c["name"], "SAM 3 prompts": ", ".join(c.get("prompts", [])) or "-"}
                      for c in S.classes]))
seg, comp, scr = S.section("segment"), S.section("compose"), S.section("screen")
display(pd.Series({
    "search engines": ", ".join(S.section("collect").get("engines", [])),
    "results per term per engine": S.section("collect")["max_results_per_query"],
    "min image side (px)": scr["min_side_px"], "blur threshold": scr["blur_min"],
    "near-duplicate (pHash bits)": scr["near_dup_hamming"],
    "SAM 3 model": seg["model_id"], "SAM 3 working size": seg["work_max_side"],
    "person / part score thresholds": f"{seg['person_threshold']} / {seg['part_threshold']}",
    "SAM 3 budget per group": seg.get("max_per_group", "all"),
    "paint order (later wins)": " > ".join(S.raw["paint_order"]),
    "output size": f"{comp['size']}x{comp['size']}", "subject fill": comp["fill"],
    "split": ", ".join(f"{k} {v:.0%}" for k, v in S.section("split")["ratios"].items()),
    "seed": S.seed}, name="value").to_frame())


## 1. Collection: search → source-page check → download

For every search result the collector opens the **source page** and keeps the image only if the group's
name is written there (in the image's own caption, the page text, or the search result title).
Pages that name three or more different groups (lists like “56个民族服饰”) are *ambiguous* and skipped.
If the original image link is blocked, the search engine's cached copy of the same image is used.

In [ ]:
logs = []
for g in GROUPS:
    p = S.raw_dir / g["code"] / "log.jsonl"
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            logs.append(json.loads(line))
if not logs:
    print("Collection has not been pulled to this computer yet (python -m ethnoseg pull raw).")
else:
    L = pd.DataFrame(logs)
    L["saved"] = L["image_id"].notna() if "image_id" in L else False
    # retries append a second record for the same URL: keep the final outcome per URL
    L = L.sort_values("saved").drop_duplicates(["group", "image_url"], keep="last")
    L["outcome"] = np.where(L["saved"], "saved", L.get("reason", pd.Series(index=L.index)).fillna("other"))
    funnel = L.pivot_table(index="group", columns="outcome", values="image_url", aggfunc="count", fill_value=0)
    funnel.insert(0, "candidates", funnel.sum(axis=1))
    funnel = funnel.reindex(CODES).fillna(0).astype(int)
    funnel.index = [LABEL[c] for c in funnel.index]
    display(funnel)
    vs = L[L["saved"]].pivot_table(index="group", columns="verify_status", values="image_url",
                                   aggfunc="count", fill_value=0).reindex(CODES).fillna(0).astype(int)
    vs.index = [LABEL[c] for c in vs.index]
    print("How the ethnic label of each saved image was verified:")
    display(vs)
    fig, ax = plt.subplots(figsize=(8, 3.4))
    order = [c for c in ["saved", "unverified", "ambiguous", "corrupt_or_not_image", "download_failed",
                         "too_small", "exact_duplicate"] if c in funnel.columns]
    funnel[order].plot.barh(stacked=True, ax=ax, colormap="tab20c", width=0.7)
    ax.set_xlabel("search results"); ax.set_title("What happened to every search result")
    ax.legend(fontsize=7, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.32), frameon=False)
    ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()


In [ ]:
# Evidence examples: the text on the source page that names the group
if logs:
    ex = (L[L["saved"]].groupby("group").head(2)[["image_id", "group", "verify_status", "evidence", "page_url"]]
          .reset_index(drop=True))
    display(ex)


## 2. Ingest and screening

Every saved image gets one row in `data/manifest.db`. Screening removes files that cannot be decoded,
images whose shorter side is under the minimum, banners/strips (extreme aspect ratio) and blurry images
(variance of the Laplacian). Then a perceptual hash (pHash) finds near-duplicates; within a group the
best copy is kept, and a picture saved under **two different groups** is removed entirely because its
label is doubtful.

In [ ]:
if M is None:
    print("Not run yet (python -m ethnoseg ingest / screen).")
else:
    M["reason_kind"] = M["reason"].fillna("").str.split(":").str[0]
    st = M.pivot_table(index="grp", columns="status", values="image_id", aggfunc="count", fill_value=0)
    st = st.reindex([c for c in CODES if c in st.index])
    st.index = [LABEL[c] for c in st.index]
    print("Current status of every image:")
    display(st)
    rej = M[M.status == "rejected"].pivot_table(index="reason_kind", columns="grp", values="image_id",
                                                aggfunc="count", fill_value=0)
    rej["total"] = rej.sum(axis=1)
    print("Why images were removed (all stages):")
    display(rej.sort_values("total", ascending=False))
    if M["blur"].notna().any():
        fig, ax = plt.subplots(figsize=(7, 2.8))
        ax.hist(np.log10(M["blur"].dropna().clip(lower=1)), bins=50, color="#6c8ebf")
        ax.axvline(np.log10(S.section("screen")["blur_min"]), color="crimson", ls="--", label="threshold")
        ax.set_xlabel("log10(sharpness)"); ax.set_ylabel("images"); ax.legend(frameon=False)
        ax.set_title("Image sharpness (variance of Laplacian)"); ax.spines[["top", "right"]].set_visible(False)
        plt.tight_layout(); plt.show()


In [ ]:
# Near-duplicate examples: removed image next to the copy that was kept
if M is not None:
    nd = M[M["reason"].fillna("").str.startswith("near_duplicate_of:")].head(4)
    items = []
    for _, r in nd.iterrows():
        keep = r["reason"].split(":")[1]
        krow = M[M.image_id == keep]
        items.append((S.raw_dir / r["file"], f"removed {r['image_id']}"))
        if len(krow):
            items.append((S.raw_dir / krow.iloc[0]["file"], f"kept {keep}"))
    show_grid(items, cols=4, title="Near-duplicates found by pHash")


## 3. SAM 3: subject and clothing-part masks (Modal GPU)

SAM 3 (`facebook/sam3` via Hugging Face `transformers`) encodes each image once and is then prompted with
text: *person* for the subject, and garment words for each class (jacket/blouse/shirt/vest → top,
skirt/trousers → bottom, dress/robe → dress, hat/headdress/headscarf → headwear, shoe/boot → footwear,
apron/belt/sash → waistwear). `ethnoseg/parts.py` turns the instance masks into one label map:

* the highest-scoring person is the subject; another person at least 30% its size → **multiple_subjects**;
* a part is used only if it overlaps the subject's own mask (hats and shoes only need to touch it), so a
  neighbouring person's clothes never attach;
* classes are painted in a fixed order so headwear, waistwear and footwear sit on top of the garment under them;
* if top **and** bottom are both clearly present, leftover *dress* pixels are given to top or bottom.

SAM 3 never decides the ethnic group; that label comes only from the source page (section 1).

In [ ]:
seg_rows = []
if M is not None and M["seg_json"].notna().any():
    for _, r in M[M["seg_json"].notna()].iterrows():
        d = json.loads(r["seg_json"]); d["grp"] = r["grp"]; seg_rows.append(d)
elif S.seg_dir.exists():                       # before compose: read the pulled SAM 3 summaries
    for p in S.seg_dir.glob("*.json"):
        d = json.loads(p.read_text(encoding="utf-8")); d["grp"] = d.get("image_id", "??")[:2]; seg_rows.append(d)
if not seg_rows:
    print("SAM 3 results not pulled yet (modal run modal_segment.py, then python -m ethnoseg pull seg).")
else:
    SG = pd.DataFrame(seg_rows)
    t = SG.pivot_table(index="grp", columns="status", values="image_id", aggfunc="count", fill_value=0)
    t = t.reindex([c for c in CODES if c in t.index]); t.index = [LABEL[c] for c in t.index]
    print("SAM 3 outcome per group:"); display(t)
    ok = SG[SG.status == "ok"]
    if len(ok) and "part_share" in ok:
        sh = pd.DataFrame(list(ok["part_share"])).assign(grp=ok["grp"].values).groupby("grp").mean()
        sh = sh.reindex([c for c in CODES if c in sh.index]); sh.index = [LABEL[c] for c in sh.index]
        fig, ax = plt.subplots(figsize=(8, 3.2))
        cols = [c for c in ["top", "bottom", "dress", "headwear", "footwear", "waistwear"] if c in sh]
        sh[cols].plot.barh(stacked=True, ax=ax, width=0.7,
                           color=[np.array(imgops.PALETTE[S.class_ids[c]]) / 255 for c in cols])
        ax.set_xlabel("average share of the subject's pixels"); ax.set_title("Clothing parts found by SAM 3")
        ax.legend(fontsize=7, ncol=6, loc="upper center", bbox_to_anchor=(0.5, -0.32), frameon=False)
        ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
        print(f"Two-piece outfits (dress pixels reassigned): {ok['two_piece'].mean():.0%} of segmented images")


## 4. Compose: white background, centred, 512×512

The subject is cut out with its mask, scaled so its longer side is 90% of 512 px, centred and placed on pure
white (255, 255, 255). The final mask holds only class ids 0–6 (skin, hair and hands are 0). Images are saved as
PNG so the white background stays exactly white. Compose also removes cut-outs that turn out to be the same
picture (pHash of the subject crop, confirmed pixel by pixel) and flags cases for the reviewer.

In [ ]:
if M is None or not (M["status"].isin(["composed", "kept"]).any() or M["fill"].notna().any()):
    print("Compose has not run yet (python -m ethnoseg compose).")
else:
    C2 = M[M["fill"].notna()]
    fig, axes = plt.subplots(1, 3, figsize=(11, 2.8))
    axes[0].hist(C2["center_dx"] * 100, bins=30, color="#82b366"); axes[0].set_title("horizontal offset (% of side)")
    axes[1].hist(C2["center_dy"] * 100, bins=30, color="#82b366"); axes[1].set_title("vertical offset (% of side)")
    axes[2].hist(C2["fill"] * 100, bins=30, color="#d6b656"); axes[2].set_title("subject size (% of side)")
    for ax in axes: ax.spines[["top", "right"]].set_visible(False)
    plt.suptitle("Placement of every composed image (brief: offset ≤ 10%, size ≥ 60%)", fontsize=10)
    plt.tight_layout(); plt.show()
    flags = Counter(f for s in C2["flags"].fillna("") for f in s.split(",") if f)
    print("Reviewer warnings:", dict(flags) or "none")
    print("Garment combinations:")
    display(C2["garment_type"].value_counts().head(10).to_frame("images"))


In [ ]:
# Examples: final image | mask overlay, two per group
if M is not None and (S.work_dir / "img").exists():
    items = []
    for code_ in CODES:
        ids = M[(M.grp == code_) & M.status.isin(["kept", "composed"])].image_id.head(2)
        for iid in ids:
            ip, mp = S.work_dir / "img" / f"{iid}.png", S.work_dir / "mask" / f"{iid}.png"
            if ip.exists() and mp.exists():
                im = np.asarray(Image.open(ip).convert("RGB")); mk = np.asarray(Image.open(mp))
                items.append((np.concatenate([im, imgops.overlay(im, mk)], axis=1), f"{iid} ({LABEL[code_]})"))
            else:
                MISSING.extend([str(ip), str(mp)])
    show_grid(items, cols=2, size=4.6, title="Composed images and their part masks", aspect=0.6)
    fig, ax = plt.subplots(figsize=(8, 0.6)); ax.axis("off")
    for i, c in enumerate(S.part_classes):
        ax.add_patch(plt.Rectangle((i * 1.3, 0), 0.3, 0.6, color=np.array(imgops.PALETTE[c["id"]]) / 255))
        ax.text(i * 1.3 + 0.38, 0.3, c["name"], va="center", fontsize=8)
    ax.set_xlim(0, 8); ax.set_ylim(0, 0.6); plt.show()


## 5. Human review

Every composed image is checked by a person in a keyboard window (`python -m ethnoseg review`): keep, or
reject with a reason (wrong parts, bad cut-out, several people, incomplete outfit, not a real photo, label
doubt). Only kept images enter the dataset. This is the manual check of the automatic SAM 3 masks that
the brief requires.

In [ ]:
if M is None or M["review"].isna().all():
    print("Review not done yet.")
else:
    R = M[M["review"].notna()]
    t = R.pivot_table(index="grp", columns="review", values="image_id", aggfunc="count", fill_value=0)
    t.index = [LABEL.get(c, c) for c in t.index]; display(t)
    print("Reject reasons:"); display(R[R.review == "reject"]["review_reason"].value_counts().to_frame("images"))
    print("Reviewers:", ", ".join(sorted(R["reviewer"].dropna().unique())))
    pending = (M.status == "composed").sum()
    print(f"Still waiting for review: {pending}")


## 6. Split, export and QC

Images from the same source page form one cluster, and a cluster never spans two splits; clusters are
assigned to train/val/test (70/15/15) per group with a fixed seed. `export` writes `dataset/` in the brief's
layout and `qc` checks every rule (sizes, class ids, pure-white background, centring, subject size, required
metadata, split leakage, minimum counts).

In [ ]:
meta_p = S.dataset_dir / "metadata.csv"
if not meta_p.exists():
    print("Dataset not exported yet (python -m ethnoseg split / export / qc).")
else:
    meta = pd.read_csv(meta_p, encoding="utf-8-sig")
    t = meta.pivot_table(index="ethnicity_code", columns="split", values="image_id", aggfunc="count", fill_value=0)
    t = t.reindex([c for c in CODES if c in t.index]); t["total"] = t.sum(axis=1)
    t.index = [LABEL[c] for c in t.index]; t.loc["Total"] = t.sum()
    display(t)
    qc_p = S.dataset_dir / "qc_report.json"
    if qc_p.exists():
        q = json.loads(qc_p.read_text(encoding="utf-8"))
        print("QC:", "PASSED" if not q["hard_failures"] else f"FAILED {q['hard_failures']}")
        print("Warnings:", q["warnings"] or "none")
        print("Largest / smallest group:", q["largest_to_smallest_group"])
        ps = pd.Series(q["pixel_share"]).drop("background", errors="ignore")
        fig, ax = plt.subplots(figsize=(6, 2.6))
        ax.bar(ps.index, ps.values * 100, color=[np.array(imgops.PALETTE[S.class_ids.get(k, 0)]) / 255 for k in ps.index])
        ax.set_ylabel("% of all pixels"); ax.set_title("Pixel share per mask class (background excluded)")
        ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()


## 7. Reproducibility

In [ ]:
import importlib.metadata as md_
def v(p):
    try: return md_.version(p)
    except Exception: return "-"
display(pd.Series({"Python (local)": platform.python_version(), "OS": platform.platform(),
                   "modal": v("modal"), "numpy": v("numpy"), "pillow": v("pillow"), "opencv-python": v("opencv-python"),
                   "GPU stage": f"Modal {S.raw['modal']['gpu']}, torch 2.10 (CUDA 12.8), transformers 5.x, {S.section('segment')['model_id']}",
                   "seed": S.seed}, name="").to_frame())
print("""Rebuild, in order (PyCharm: Run configurations 01-13):
  modal run modal_collect.py            # (+ --retry-failed) search, verify, download
  python -m ethnoseg pull raw
  python -m ethnoseg ingest
  python -m ethnoseg screen
  modal run modal_segment.py            # SAM 3 on Modal GPU
  python -m ethnoseg pull seg
  python -m ethnoseg compose
  python -m ethnoseg review             # by a person
  python -m ethnoseg split && python -m ethnoseg export && python -m ethnoseg qc && python -m ethnoseg stats""")
if MISSING:
    print(f"\n{len(MISSING)} example image files were not found on this computer (examples left blank).")
